# Context Engineering: Managing the Context Window

The context window is the most constrained resource in LLM applications. Context engineering — deciding what goes in, what gets compressed, and what gets dropped — directly determines response quality, cost, and latency.

## Context Window Economics

For a 128K token context window at typical API pricing:
- Input tokens cost ~$3/million (Claude Sonnet pricing)
- A full 128K context costs $0.384 per call
- With prompt caching, repeated context blocks cost ~$0.30/million

At scale, context engineering is a cost engineering problem. A RAG pipeline that retrieves 20 irrelevant chunks (5K tokens) before the actual context wastes $0.015 per call — at 1M calls/day, that is $15,000/day in unnecessary context.

Beyond cost: model quality degrades with irrelevant context. The 'lost in the middle' phenomenon (Liu et al. 2023) showed models pay less attention to content in the middle of long contexts. Keeping context focused improves both quality and cost.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import math

@dataclass
class ContextBlock:
    content: str
    priority: int  # 1=highest, 5=lowest
    block_type: str  # 'system', 'instruction', 'history', 'retrieval', 'tool_result'
    token_estimate: int = 0
    cacheable: bool = False

    def __post_init__(self):
        if self.token_estimate == 0:
            self.token_estimate = len(self.content.split()) * 4 // 3  # rough estimate

class ContextManager:
    PRIORITY_ORDER = ['system', 'instruction', 'tool_result', 'retrieval', 'history']

    def __init__(self, max_tokens: int = 8000, reserve_output: int = 1000):
        self.max_tokens = max_tokens
        self.reserve_output = reserve_output
        self.budget = max_tokens - reserve_output
        self.blocks: list = []

    def add(self, block: ContextBlock):
        self.blocks.append(block)

    def build(self) -> tuple:
        # Sort by priority (lower number = higher priority)
        sorted_blocks = sorted(self.blocks, key=lambda b: (b.priority, self.PRIORITY_ORDER.index(b.block_type)
                                                           if b.block_type in self.PRIORITY_ORDER else 99))
        included = []
        excluded = []
        used_tokens = 0
        for block in sorted_blocks:
            if used_tokens + block.token_estimate <= self.budget:
                included.append(block)
                used_tokens += block.token_estimate
            else:
                excluded.append(block)
        context = '\n\n'.join(b.content for b in included)
        return context, {'included': len(included), 'excluded': len(excluded),
                          'tokens_used': used_tokens, 'budget': self.budget,
                          'utilization': used_tokens/self.budget}

    def cost_estimate(self, price_per_million: float = 3.0) -> float:
        total_tokens = sum(b.token_estimate for b in self.blocks)
        return total_tokens * price_per_million / 1e6

cm = ContextManager(max_tokens=4000, reserve_output=500)
cm.add(ContextBlock('You are a helpful coding assistant.', priority=1, block_type='system', cacheable=True))
cm.add(ContextBlock('Help me debug this Python error.', priority=1, block_type='instruction'))
cm.add(ContextBlock('Retrieval result 1: ' + 'relevant code snippet ' * 50, priority=2, block_type='retrieval'))
cm.add(ContextBlock('Retrieval result 2: ' + 'less relevant docs ' * 80, priority=3, block_type='retrieval'))
cm.add(ContextBlock('Retrieval result 3: ' + 'marginally relevant ' * 100, priority=4, block_type='retrieval'))
cm.add(ContextBlock('Previous conversation history ' * 30, priority=3, block_type='history'))

context, stats = cm.build()
print('Context build stats:')
for k, v in stats.items():
    print(f'  {k}: {v:.2f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'Estimated cost: ${cm.cost_estimate():.6f}')

## Retrieval-Augmented Context

For knowledge-intensive applications, relevant documents are retrieved and injected into the context. Key decisions:

**Chunk size**: smaller chunks (128-256 tokens) are more precise but require more retrieval calls; larger chunks (512-1024 tokens) provide more context per retrieval.

**Retrieval count**: more chunks = more coverage but more noise. 3-10 chunks is typical.

**Reranking**: a lightweight cross-encoder model reranks retrieved chunks for relevance before injecting into context. Significantly improves quality.

**Context placement**: place the most relevant content near the beginning or end of the context window (avoiding 'lost in the middle').

In [ ]:
def retrieve_and_rank(query: str, documents: list, top_k: int = 3) -> list:
    # Simple TF-IDF-like scoring
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        doc_words = set(doc['content'].lower().split())
        overlap = len(query_words & doc_words)
        idf_boost = math.log(1 + len(doc['content'].split()) / 50)  # prefer substantial docs
        score = overlap * idf_boost
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [doc for _, doc in scored[:top_k]]

docs = [
    {'id': 'doc1', 'content': 'Python IndexError occurs when you access a list index out of range. Check list length before accessing.'},
    {'id': 'doc2', 'content': 'Python list slicing with negative indices works from the end of the list.'},
    {'id': 'doc3', 'content': 'Common Python debugging: use pdb.set_trace() or breakpoint() to step through code interactively.'},
    {'id': 'doc4', 'content': 'Machine learning model training with PyTorch requires careful gradient management.'},
    {'id': 'doc5', 'content': 'Python exceptions: IndexError, KeyError, and TypeError are the most common runtime errors.'},
]

results = retrieve_and_rank('help debugging IndexError in Python list', docs, top_k=3)
print('Top retrieved documents:')
for i, doc in enumerate(results, 1):
    print(f'  {i}. [{doc["id"]}] {doc["content"][:80]}')